# 01: Data Preprocessing

### Data Download for Clinical, Microenvironment, and Expression data
1. Go to https://portal.gdc.cancer.gov/analysis_page?app=CohortBuilder&tab=general
    - Select Program: TCGA
    - "# CASES" -> "Table View" -> "TSV"
    - Download to `data_raw/`
2. Go to "Repository" with the previous TCGA selection still active.
2. Make the following selections: 
    - Experimental Strategy: RNA-Seq
    - Access: open
    - Tumor Descriptor: primary, not applicable
    - Specimen Type: solid tissue
    - Preservation Method: unknown
3. "Add All Files to Cart". Go to "Cart". 
4. Place the following downloads in the `data_raw/unknown` folder.
    - Download Associated Data
        - Clinical: TSV
        - Biospecimen: TSV
        - Sample Sheet
    - Unzip these files
5. Place the following downloads in the `data_raw/unknown/rnaseq` folder.
    - Download Cart
        - Manifest
4. "Remove from Cart" -> "All Files"
4. Go back to downloads, select the same filters but change
    - Preservation Method: all except unknown (oct and ffpe)
5. Repeat the download, placing in `data_raw/other` and `data_raw/other/rnaseq`.
6. Use the GDC client CLI to download both manifests (~60GB)
```
cd data_raw/unknown/rnaseq
gdc-client download -m ../<your_unknown_manifest>.txt
cd ../../other/rnaseq
gdc-client download -m ../<your_other_manifest>.txt
```

### Data Download for SNV and SCNA data
This data is under access controls. Please apply online and refer to the cited studies in our supplementary material for compiling this data.

In [84]:
import os
import pandas as pd
import numpy as np

# Replace these paths to run for both the "unknown" and "other" downloads
in_dir = 'data_raw/unknown'
out_dir = 'data_intermediate/unknown'

# Replace these with your folder names
tcga_cases_tsv = 'data_raw/cases.tsv'
tcga_clinical_tsv = os.path.join(in_dir, 'clinical.cart.2025-01-17/clinical.tsv')
tcga_sample_tsv = os.path.join(in_dir, 'biospecimen.cart.2025-01-17/sample.tsv')
tcga_slide_tsv = os.path.join(in_dir, 'biospecimen.cart.2025-01-17/slide.tsv')
tcga_file_tsv = os.path.join(in_dir, 'gdc_sample_sheet.2025-01-17.tsv')

In [85]:
tcga_cases_df = pd.read_csv(tcga_cases_tsv, sep='\t')
tcga_clinical_df = pd.read_csv(tcga_clinical_tsv, sep='\t')
tcga_sample_df = pd.read_csv(tcga_sample_tsv, sep='\t')
tcga_slide_df = pd.read_csv(tcga_slide_tsv, sep='\t')
tcga_file_df = pd.read_csv(tcga_file_tsv, sep='\t')

/var/folders/_6/468mk1cx3cb080z2r5gq0k_r0000gn/T/ipykernel_71551/206510135.py:1: DtypeWarning: Columns (11,13,15,17,19,21,23,25,27,29,33,35,37,39,41,43,45,47,61) have mixed types. Specify dtype option on import or set low_memory=False.
  tcga_cases_df = pd.read_csv(tcga_cases_tsv, sep='\t')


In [86]:
selected_case_cols = [
    'case_id',
    'submitter_id',
    'primary_site',
]
selected_clinical_cols = [
    'case_id',
    'case_submitter_id',
    'project_id',
    'age_at_diagnosis',
    'gender',
    'year_of_birth',
    'race',
    'ajcc_pathologic_stage',
    'ajcc_clinical_stage',
    'ann_arbor_clinical_stage',
    'figo_stage',
]
selected_survival_cols = [
    'case_id',
    'case_submitter_id',
    'vital_status',
    'days_to_death',
    'days_to_last_follow_up',
]
selected_sample_cols = [
    'case_id',
    'case_submitter_id',
    'sample_id',
    'sample_submitter_id',
    'days_to_collection',
    'sample_type',
]
selected_slide_cols = [
    'case_id',
    'case_submitter_id',
    'sample_id',
    'sample_submitter_id',
    'slide_submitter_id',
    'percent_neutrophil_infiltration',
    'percent_monocyte_infiltration',
    'percent_normal_cells',
    'percent_tumor_nuclei',
    'percent_lymphocyte_infiltration',
    'percent_stromal_cells',
    'percent_tumor_cells',
]
study_names = {
    "LAML": "Acute Myeloid Leukemia",
    "ACC": "Adrenocortical carcinoma",
    "BLCA": "Bladder Urothelial Carcinoma",
    "LGG": "Brain Lower Grade Glioma",
    "BRCA": "Breast invasive carcinoma",
    "CESC": "Cervical squamous cell carcinoma and endocervical adenocarcinoma",
    "CHOL": "Cholangiocarcinoma",
    "LCML": "Chronic Myelogenous Leukemia",
    "COAD": "Colon adenocarcinoma",
    "CNTL": "Controls",
    "ESCA": "Esophageal carcinoma",
    "FPPP": "FFPE Pilot Phase II",
    "GBM": "Glioblastoma multiforme",
    "HNSC": "Head and Neck squamous cell carcinoma",
    "KICH": "Kidney Chromophobe",
    "KIRC": "Kidney renal clear cell carcinoma",
    "KIRP": "Kidney renal papillary cell carcinoma",
    "LIHC": "Liver hepatocellular carcinoma",
    "LUAD": "Lung adenocarcinoma",
    "LUSC": "Lung squamous cell carcinoma",
    "DLBC": "Lymphoid Neoplasm Diffuse Large B-cell Lymphoma",
    "MESO": "Mesothelioma",
    "MISC": "Miscellaneous",
    "OV": "Ovarian serous cystadenocarcinoma",
    "PAAD": "Pancreatic adenocarcinoma",
    "PCPG": "Pheochromocytoma and Paraganglioma",
    "PRAD": "Prostate adenocarcinoma",
    "READ": "Rectum adenocarcinoma",
    "SARC": "Sarcoma",
    "SKCM": "Skin Cutaneous Melanoma",
    "STAD": "Stomach adenocarcinoma",
    "TGCT": "Testicular Germ Cell Tumors",
    "THYM": "Thymoma",
    "THCA": "Thyroid carcinoma",
    "UCS": "Uterine Carcinosarcoma",
    "UCEC": "Uterine Corpus Endometrial Carcinoma",
    "UVM": "Uveal Melanoma",
}

In [87]:
print('SURVIVAL')
tcga_survival_df = tcga_clinical_df[selected_survival_cols]
display(tcga_survival_df.head())
print('CASES')
tcga_cases_df = tcga_cases_df[selected_case_cols]
display(tcga_cases_df.head())
print('CLINICAL')
tcga_clinical_df = tcga_clinical_df[selected_clinical_cols]
display(tcga_clinical_df.head())
print('SAMPLE')
tcga_sample_df = tcga_sample_df[selected_sample_cols]
display(tcga_sample_df.head())
print('SLIDE')
tcga_slide_df = tcga_slide_df[selected_slide_cols]
display(tcga_slide_df.head())
print('RNASEQ FILES')
display(tcga_file_df.head())

SURVIVAL


,case_id,case_submitter_id,vital_status,days_to_death,days_to_last_follow_up
0,0011a67b-1ba9-4a32-a6b8-7850759a38cf,TCGA-DC-6158,Dead,334,216.0
1,0011a67b-1ba9-4a32-a6b8-7850759a38cf,TCGA-DC-6158,Dead,334,216.0
2,001944e5-af34-4061-9c09-bb9ea346f6fd,TCGA-HQ-A5ND,Dead,274,'--
3,001944e5-af34-4061-9c09-bb9ea346f6fd,TCGA-HQ-A5ND,Dead,274,'--
4,001ad307-4ad3-4f1d-b2fc-efc032871c7e,TCGA-HT-A614,Alive,'--,82.0


CASES


,case_id,submitter_id,primary_site
0,4298ccdb-2e6d-4267-822d-75b021364084,TCGA-B0-4710,Kidney
1,a2663a86-a006-4867-9e88-2b523df48303,TCGA-B8-A54K,Kidney
2,439794a8-51bd-4c70-968c-34cf26b90148,TCGA-B0-5113,Kidney
3,e865d40a-9989-436c-8426-88cc84c863e8,TCGA-CJ-5689,Kidney
4,305eaef4-4644-46e3-a696-d2e4a972f691,TCGA-CZ-4865,Kidney


CLINICAL


...

,case_id,case_submitter_id,sample_id,sample_submitter_id,days_to_collection,sample_type
0,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,5e797e09-26d0-42f4-838f-3b05e489e6ec,TCGA-AA-3561-01A,'--,Primary Tumor
1,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,699f2743-46f7-4967-b670-30c93a59ab0f,TCGA-AA-3561-01Z,'--,Primary Tumor
2,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,69cff594-84a3-4d68-9edd-025b7d540535,TCGA-AA-3561-10A,'--,Blood Derived Normal
3,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,eb29b3dc-6291-490c-b143-cd7c235904e2,TCGA-AA-3561-10B,'--,Blood Derived Normal
4,77200abe-4a09-4a4a-bb45-60d9b96d4998,TCGA-EL-A3H2,405fbe40-5934-431a-8806-325ac0d788ed,TCGA-EL-A3H2-11A,3292,Solid Tissue Normal


SLIDE


,case_id,case_submitter_id,sample_id,sample_submitter_id,slide_submitter_id,percent_neutrophil_infiltration,percent_monocyte_infiltration,percent_normal_cells,percent_tumor_nuclei,percent_lymphocyte_infiltration,percent_stromal_cells,percent_tumor_cells
0,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,5e797e09-26d0-42f4-838f-3b05e489e6ec,TCGA-AA-3561-01A,TCGA-AA-3561-01A-01-BS1,'--,'--,0.0,90.0,'--,10.0,90.0
1,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,5e797e09-26d0-42f4-838f-3b05e489e6ec,TCGA-AA-3561-01A,TCGA-AA-3561-01A-01-TS1,'--,'--,0.0,90.0,'--,20.0,80.0
2,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,699f2743-46f7-4967-b670-30c93a59ab0f,TCGA-AA-3561-01Z,TCGA-AA-3561-01Z-00-DX1,'--,'--,'--,'--,'--,'--,'--
3,77200abe-4a09-4a4a-bb45-60d9b96d4998,TCGA-EL-A3H2,405fbe40-5934-431a-8806-325ac0d788ed,TCGA-EL-A3H2-11A,TCGA-EL-A3H2-11A-01-TS1,0.0,5.0,100.0,0.0,2.0,0.0,0.0
4,77200abe-4a09-4a4a-bb45-60d9b96d4998,TCGA-EL-A3H2,64e11d42-fec9-47e4-a6a6-8df2a6bfd79b,TCGA-EL-A3H2-01A,TCGA-EL-A3H2-01A-01-TS1,0.0,0.0,0.0,95.0,0.0,5.0,95.0


RNASEQ FILES


,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Sample Type
0,98e7190a-4d3a-4fdb-a021-c482f87e65f2,da658a99-8972-473e-9754-9712be0b1b25.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-R6-A6XQ,TCGA-R6-A6XQ-01B,Primary Tumor
1,3da6be11-66bd-437b-9ce6-87e63beedbec,20f5e7d5-0faf-47f6-9e7c-ecef50a58288.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-R6-A6KZ,TCGA-R6-A6KZ-01A,Primary Tumor
2,79ce65af-7af4-45bd-93ea-ce87652a1f8d,25961c5e-75a1-4f22-b3b8-6f373dad4d98.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-L5-A43E,TCGA-L5-A43E-01A,Primary Tumor
3,5d321eee-f8c4-4460-b4ee-4a3f016206e4,93d85034-6192-4ee1-a0ec-8126e8c049a4.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-JY-A6F8,TCGA-JY-A6F8-01A,Primary Tumor
4,c702ce91-1d1d-4d1c-ba9b-1dbe937425e2,a3e18366-19fa-4d0a-96e8-f15af6393a87.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-RE-A7BO,TCGA-RE-A7BO-01A,Primary Tumor


In [88]:
# Compile clinical covariates
my_clinical_df = tcga_clinical_df.merge(tcga_cases_df, on='case_id')
my_clinical_df['disease_type'] = tcga_clinical_df['project_id'].map(lambda s: s.split('-')[1])
my_clinical_df.drop(columns=['case_submitter_id', 'project_id'], inplace=True)
my_clinical_df.replace("'--", None, inplace=True)

stage_features = [
    'ajcc_pathologic_stage',
    'ajcc_clinical_stage',
    'ann_arbor_clinical_stage',
    'figo_stage',
]
stages = []
for i, (ajcc_path, ajcc_clin, ann, figo) in my_clinical_df[stage_features].iterrows():
    def get_stage(s):
        if "IV" in s:
            return 4.
        elif "III" in s:
            return 3.
        elif "II" in s:
            return 2.
        elif "I" in s:
            return 1.
        else:
            return None
    if ajcc_path is not None:
        stage = get_stage(ajcc_path)
    elif ajcc_clin is not None:
        stage = get_stage(ajcc_clin)
    elif ann is not None:
        stage = get_stage(ann)
    elif figo is not None:
        stage = get_stage(figo)
    else:
        stage = None
    stages.append(stage)
my_clinical_df['stage'] = stages 
my_clinical_df.drop(columns=stage_features, inplace=True)

my_clinical_df.head()

,case_id,age_at_diagnosis,gender,year_of_birth,race,submitter_id,primary_site,disease_type,stage
0,0011a67b-1ba9-4a32-a6b8-7850759a38cf,25842,male,1940,white,TCGA-DC-6158,Rectum,READ,1.0
1,0011a67b-1ba9-4a32-a6b8-7850759a38cf,25842,male,1940,white,TCGA-DC-6158,Rectum,READ,1.0
2,001944e5-af34-4061-9c09-bb9ea346f6fd,28555,male,None,not reported,TCGA-HQ-A5ND,Bladder,BLCA,4.0
3,001944e5-af34-4061-9c09-bb9ea346f6fd,28555,male,None,not reported,TCGA-HQ-A5ND,Bladder,BLCA,4.0
4,001ad307-4ad3-4f1d-b2fc-efc032871c7e,17392,male,1966,white,TCGA-HT-A614,Brain,LGG,NaN


In [36]:
# Compile samples
# Only use the first tumor and healthy sample from each patient
my_sample_df = tcga_sample_df[tcga_sample_df['sample_submitter_id'].map(lambda s: '-11A' in s or '-01A' in s)]
my_sample_df = my_sample_df.merge(tcga_slide_df.drop(columns=['case_id', 'case_submitter_id', 'sample_submitter_id', 'slide_submitter_id']), on='sample_id')
my_sample_df.drop_duplicates(subset='sample_id', inplace=True)  # Multiple slides, just take the first
my_sample_df.replace("'--", None, inplace=True)
my_sample_df.head()

,case_id,case_submitter_id,sample_id,sample_submitter_id,days_to_collection,sample_type,percent_neutrophil_infiltration,percent_monocyte_infiltration,percent_normal_cells,percent_tumor_nuclei,percent_lymphocyte_infiltration,percent_stromal_cells,percent_tumor_cells
0,01ad5016-f691-4bca-82a0-910429d8d25b,TCGA-AA-3561,5e797e09-26d0-42f4-838f-3b05e489e6ec,TCGA-AA-3561-01A,None,Primary Tumor,None,None,0.0,90.0,None,10.0,90.0
2,77200abe-4a09-4a4a-bb45-60d9b96d4998,TCGA-EL-A3H2,405fbe40-5934-431a-8806-325ac0d788ed,TCGA-EL-A3H2-11A,3292,Solid Tissue Normal,0.0,5.0,100.0,0.0,2.0,0.0,0.0
3,77200abe-4a09-4a4a-bb45-60d9b96d4998,TCGA-EL-A3H2,64e11d42-fec9-47e4-a6a6-8df2a6bfd79b,TCGA-EL-A3H2-01A,3292,Primary Tumor,0.0,0.0,0.0,95.0,0.0,5.0,95.0
4,25eb6a8a-5bd9-480d-baab-6113e1b3be51,TCGA-L5-A88S,0ecc7f6e-65a4-42e0-9bb0-d787beecd88d,TCGA-L5-A88S-01A,155,Primary Tumor,0.0,0.0,30.0,75.0,2.0,0.0,70.0
5,25eb6a8a-5bd9-480d-baab-6113e1b3be51,TCGA-L5-A88S,de23dd58-a9db-44f3-b1ff-fa96429e79aa,TCGA-L5-A88S-11A,155,Solid Tissue Normal,0.0,0.0,100.0,0.0,4.0,0.0,0.0


In [58]:
# Survival
my_survival_df = tcga_survival_df[['case_id', 'case_submitter_id', 'vital_status', 'days_to_death', 'days_to_last_follow_up']]
my_survival_df['died'] = my_survival_df['vital_status'] == 'Dead'
my_survival_df['time'] = my_survival_df['days_to_death']
my_survival_df.replace("'--", None, inplace=True)
my_survival_df['time'].fillna(my_survival_df['days_to_last_follow_up'], inplace=True)
my_survival_df.drop(columns=['vital_status', 'days_to_death', 'days_to_last_follow_up'], inplace=True)
my_survival_df.drop_duplicates(inplace=True)
my_survival_df

,case_id,case_submitter_id,died,time
0,0011a67b-1ba9-4a32-a6b8-7850759a38cf,TCGA-DC-6158,True,334
2,001944e5-af34-4061-9c09-bb9ea346f6fd,TCGA-HQ-A5ND,True,274
4,001ad307-4ad3-4f1d-b2fc-efc032871c7e,TCGA-HT-A614,False,82.0
6,001cef41-ff86-4d3f-a140-a647ac4b10a1,TCGA-E2-A1IU,False,337.0
8,001e0309-9c50-42b0-9e38-347883ee2cd3,TCGA-D1-A17N,False,46.0
...,...,...,...,...
10031,ffb53b81-c1ed-473c-bdcd-b325b02e22ae,TCGA-98-8020,True,84
10033,ffcec8e5-9fd3-4b42-a7cb-74761f713cf4,TCGA-HT-7606,False,526.0
10035,ffcf851d-7fa1-4b45-911a-a3fbd74c253a,TCGA-CN-6016,False,1443.0
10037,ffedc8be-1056-4205-b9d9-99b5bdb872db,TCGA-N9-A4Q4,False,1089.0


In [39]:
# Gene expression